In this notebook we will try to train the model using news and making sentiment of them using data source from gdeltproject.org.

**Fetching news**

In [ ]:
import os
import requests
import zipfile
import io
import pandas as pd
import datetime
import time

In [ ]:
DATA_FOLDER = "gdelt_gold_filtered"
os.makedirs(DATA_FOLDER, exist_ok=True)

In [ ]:
def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days) + 1):
        yield start_date + datetime.timedelta(n)

In [ ]:
def download_and_filter_gdelt(date):
    date_str = date.strftime("%Y%m%d")
    url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
    try:
        print(f"Downloading {url} ...")
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f"No data for {date_str} (status {r.status_code})")
            return False

        z = zipfile.ZipFile(io.BytesIO(r.content))
        filename = z.namelist()[0]
        with z.open(filename) as f:
            columns = [
                "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",
                "Actor1Code", "Actor1Name", "Actor1CountryCode", "Actor1KnownGroupCode",
                "Actor1EthnicCode", "Actor1Religion1Code", "Actor1Religion2Code",
                "Actor1Type1Code", "Actor1Type2Code", "Actor1Type3Code",
                "Actor2Code", "Actor2Name", "Actor2CountryCode", "Actor2KnownGroupCode",
                "Actor2EthnicCode", "Actor2Religion1Code", "Actor2Religion2Code",
                "Actor2Type1Code", "Actor2Type2Code", "Actor2Type3Code",
                "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
                "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
                "NumArticles", "AvgTone", "Actor1Geo_Type", "Actor1Geo_FullName",
                "Actor1Geo_CountryCode", "Actor1Geo_ADM1Code", "Actor1Geo_Lat",
                "Actor1Geo_Long", "Actor1Geo_FeatureID", "Actor2Geo_Type",
                "Actor2Geo_FullName", "Actor2Geo_CountryCode", "Actor2Geo_ADM1Code",
                "Actor2Geo_Lat", "Actor2Geo_Long", "Actor2Geo_FeatureID",
                "ActionGeo_Type", "ActionGeo_FullName", "ActionGeo_CountryCode",
                "ActionGeo_ADM1Code", "ActionGeo_Lat", "ActionGeo_Long",
                "ActionGeo_FeatureID", "DATEADDED", "SOURCEURL"
            ]
            df = pd.read_csv(f, sep="\t", header=None, names=columns, dtype=str)

        keyword_filter = df['SOURCEURL'].str.contains("gold", case=False, na=False) | \
                         df['SOURCEURL'].str.contains("goldprice", case=False, na=False) | \
                         df['SOURCEURL'].str.contains("gold-price", case=False, na=False) | \
                         df['EventCode'].str.contains("gold", case=False, na=False)

        filtered_df = df[keyword_filter]

        output_path = os.path.join(DATA_FOLDER, f"{date_str}_gold_filtered.csv")

        if not filtered_df.empty:
            filtered_df.to_csv(output_path, index=False)
            print(f"Saved {len(filtered_df)} gold-related events for {date_str}")
        else:
            pd.DataFrame([{"Date": date_str, "Note": "No gold-related news"}]).to_csv(output_path, index=False)
            print(f"No gold-related events found for {date_str}")

        return True

    except Exception as e:
        print(f"Error processing {date_str}: {e}")
        return False


start_date = datetime.date(2014, 1, 1)
end_date = datetime.date.today()

for single_date in daterange(start_date, end_date):
    download_and_filter_gdelt(single_date)
    time.sleep(1)  


Saved 133 gold-related events for 20140101
Saved 210 gold-related events for 20140102
Saved 269 gold-related events for 20140103
Saved 84 gold-related events for 20140104
Saved 118 gold-related events for 20140105
Saved 319 gold-related events for 20140106
Saved 278 gold-related events for 20140107
Saved 356 gold-related events for 20140108
Saved 355 gold-related events for 20140109
Saved 341 gold-related events for 20140110
Saved 196 gold-related events for 20140111
Saved 263 gold-related events for 20140112
Saved 649 gold-related events for 20140113
Saved 640 gold-related events for 20140114
Saved 479 gold-related events for 20140115
Saved 314 gold-related events for 20140116
Saved 58 gold-related events for 20140117
Saved 15 gold-related events for 20140118
Saved 17 gold-related events for 20140119
Saved 86 gold-related events for 20140120
Saved 52 gold-related events for 20140121
Saved 70 gold-related events for 20140122
No data for 20140123 (status 404)
No data for 20140124 (statu